# topk-predictions — worked example 1: Top-3 accuracy from logits

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `topk-predictions`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`logits.topk(k, dim=-1)` returns a named tuple with `.values` and `.indices`; `.indices` holds the `k` highest-scoring class ids per row. A sample counts as a top-k hit if its true label is among those `k` indices. Broadcasting the label against the index axis and reducing with `.any` gives a per-sample hit mask.

## Worked solution

We compute top-3 classification accuracy for a batch of logits.

1. `logits.topk(3, dim=-1)` finds the 3 largest logits per row; we take `.indices`, shape `(B, 3)`.
2. `labels.unsqueeze(-1)` reshapes the labels to `(B, 1)` so they broadcast against the `(B, 3)` indices.
3. The equality `topk.indices == labels.unsqueeze(-1)` is `True` wherever a top-3 slot matches the true label; `.any(dim=-1)` collapses the 3 slots to one boolean per sample.
4. Casting to float and taking the mean gives the fraction of samples whose label landed in the top 3.

We print the accuracy and confirm it lies in [0, 1].

In [ ]:
import torch as t

t.manual_seed(0)

def top3_accuracy(logits: t.Tensor, labels: t.Tensor) -> t.Tensor:
    topk = logits.topk(3, dim=-1)
    correct = (topk.indices == labels.unsqueeze(-1)).any(dim=-1)
    return correct.float().mean()

logits = t.randn(8, 10)
labels = t.randint(0, 10, (8,))
acc = top3_accuracy(logits, labels)
print('top-3 acc:', round(acc.item(), 3))
print('in range:', 0.0 <= acc.item() <= 1.0)